# Two algorithms regarding Sampling

In this notebook, we present several relatively simple algorithms that were used in our study of the influence of disorder on the phase transition of the transverse Ising model, but which may also prove useful in other contexts.


# I. Ensuring the thermalization of a Markov chain

As we emphasized in the previous notebook, it is essential to ensure that the states of the sampling chains used to evaluate a physical observable are “sufficiently close” to the desired target distribution. This allows us to guarantee that the value of the observable computed from these samples, which contributes to the estimation of the observable in the target state, does not deviate excessively from its true value.

One possible approach is to burn a certain number of samples, meaning that we only begin to take into account the values of observables computed from these samples in the estimation of the observable in the target state after a given point in the chain.

One of the objectives of the algorithm below is therefore to determine this number of samples to discard.

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import netket as nk
import netket_foundation as nkf
import flax

def thermalizing(etat_foundation, sampler, operator, max_chain_length_init: int, plot: bool, seuil1: float, seuil2: float, nb_tentatives: int):
    current_max = max_chain_length_init
    stats, hist_data = None, None

    for tentative in range(nb_tentatives):
        print(f"Tentative {tentative+1} avec max_chain_length={current_max}")

        # Bloc 1 for commentaries
        _, hist_data = etat_foundation.check_mc_convergence(
            operator,
            min_chain_length=50,
            max_chain_length=current_max,
            plot=plot
        )
        #Bloc 2 :
        errors = hist_data["error_of_mean"]
        r_hats = hist_data["R_hat"]
        premier_indice = None
        for i, (err, rhat) in enumerate(zip(errors, r_hats)):
            if abs(err) < seuil1 and abs(rhat - 1.0) < seuil2:
                premier_indice = i
                break
        
        if premier_indice is not None:
            break  
        
        print(f"  Seuil non atteint (error_of_mean finale = {errors[-1]:.2e}), doublement de max_chain_length...")
        current_max *= 2
    else:
        raise RuntimeError(
            f"Thermalisation non atteinte après {nb_tentatives} tentatives.\n"
            f"  max_chain_length atteint = {current_max}\n"
            f"  error_of_mean finale     = {errors[-1]:.2e} > seuil = {seuil:.2e}\n"
            f"  tau_corr final           = {hist_data['tau_corr_acf'][-1]:.2e}\n"
            f"  R_hat final              = {hist_data['R_hat'][-1]:.4f}\n"
            f"  sweep_size final         = {hist_data['sweep_size'][-1]}\n"
            f"  → Vérifiez votre modèle ou assouplissez le seuil."
        )
    tau_corr   = hist_data["tau_corr_acf"]
    R_hat      = hist_data["R_hat"]
    sweep_size = hist_data["sweep_size"]
    print(f"Convergence atteinte à l'itération {premier_indice}")
    print(f"  error_of_mean au seuil = {errors[premier_indice]:.2e}")
    print(f"  tau_corr final         = {tau_corr[-1]:.2e}")
    print(f"  R_hat final            = {R_hat[-1]:.4f}")
    print(f"  sweep_size final       = {sweep_size[-1]}")
    
    #Bloc 3
    vstate_thermalized = nk.vqs.MCState(
        sampler=sampler,
        model=etat_foundation.model,
        variables=etat_foundation.variables,
        n_samples=current_max,
        n_discard_per_chain=premier_indice
    )
    convergence_info = {
        "premier_indice" : premier_indice,
        "error_of_mean"  : errors[premier_indice],
        "tau_corr"       : tau_corr[-1],
        "R_hat"          : R_hat[-1],
        "sweep_size"     : sweep_size[-1],
        "max_chain_length_utilise" : current_max,
    }
    return hist_data, convergence_info, vstate_thermalized

The principle of the algorithm is that the user provides as input two threshold values, a number of trials, and a maximum initial chain length. At each iteration, the maximum chain length is doubled, and the algorithm uses the History_Dict object from the check_mc_convergence() method to find the smallest index, if it exists, such that the relative error and the r-hat fall below the specified threshold. If such an index exists, it is stored and defines the number of samples to be burned at the beginning. Otherwise, the maximum chain length is doubled, and the algorithm regenerates the samples until an index is found for which the relative error is below the threshold.

Finally, the algorithm returns the thermalized MCState along with various statistical objects.

Code explanation:

Block 1 : The function check_mc_convergence provides diagnostic statistics for simulated Markov chains. To this end, it runs Markov chains of increasing length in order to compare the associated statistics. The chain length ranges from min_chain_length to max_chain_length. The function also takes several additional arguments:
-an operator (for example, the energy),
-a histogram, hist_data, which stores the values obtained from evaluating the operator at each sample of the Markov chain under consideration. This histogram also includes values of key diagnostics for the Markov chain (such as the autocorrelation rate, the R-hat
statistic, and the sweep_size, i.e., the number of samples skipped before re-evaluating the operator; this helps ensure approximate independence of samples when the sweep_size is on the order of the autocorrelation rate).


Block 2: The first index is computed such that both the relative error and the R-hat statistic fall below user-defined thresholds. If such an index exists, it is selected as the starting point for sampling. Otherwise, as previously described, the maximum chain length is doubled and the process is repeated.

Block 3: the first index is taken to override (or supersede) the preceding ones.

# II. Reusing an MCState to initialize the sampling

When one aims to evaluate a physical observable on a given state, sampling is required. To avoid a random initialization of the sampling chains, it can be desirable to start from a predefined state. The short algorithm below does exactly this: it returns an MCState whose sampler state matches that of a previously obtained MCState.

In [ ]:

def foundation_state_get_state1(state : nk.vqs.MCState):
    model=state.model
    sampler=state.sampler
    n_samples=state.n_samples
    n_discard_per_chain=state.n_discard_per_chain
    variables=state.variables
    mc_state=nk.vqs.MCState(
        sampler=sampler,
        model=model,
        variables=variables,
        n_samples=n_samples,
        n_discard_per_chain=n_discard_per_chain
    )
    #Block 1
    mc_state.sampler_state = mc_state.sampler.init_state(
    mc_state.sampler,
    mc_state.model,
    mc_state.variables,
    seed=0
    )

    mc_state.sampler_state = mc_state.sampler_state.replace(
    initial_config = mc_state.sampler_state.σ   
    )
    return mc_state

Block 1: The sampler state of the MCState is updated to match that of the previous MCState.